In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/intern_sentiment_dataset.csv")

label_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

df["label"] = df["sentiment"].map(label_map)

df = df[["feedback", "label"]]

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

dataset = Dataset.from_pandas(df)

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

def tokenize(batch):
    return tokenizer(
        batch["feedback"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tokenize, batched=True)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/67529 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_output",

    save_strategy="epoch",
    eval_strategy="epoch",   # ✅ THIS instead of evaluation_strategy

    num_train_epochs=3,

    save_total_limit=2,
    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    weight_decay=0.01,
)

In [ ]:
import transformers
print(transformers.__version__)

5.12.0


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.554600,0.543581
2,0.449690,0.582399


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.554600,0.543581
2,0.449690,0.582399
3,0.304844,0.678936


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=10131, training_loss=0.4530370326518024, metrics={'train_runtime': 7359.8548, 'train_samples_per_second': 22.021, 'train_steps_per_second': 1.377, 'total_flos': 2.1321264248879616e+16, 'train_loss': 0.4530370326518024, 'epoch': 3.0})

In [ ]:
model.save_pretrained("bert_sentiment_model")
tokenizer.save_pretrained("bert_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_sentiment_model/tokenizer_config.json',
 'bert_sentiment_model/tokenizer.json')

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model.save_pretrained("/content/drive/MyDrive/my_model")
tokenizer.save_pretrained("/content/drive/MyDrive/my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/my_model/tokenizer_config.json',
 '/content/drive/MyDrive/my_model/tokenizer.json')

	zip warning: name not matched: final_model

zip error: Nothing to do! (try: zip -r final_model.zip . -i final_model)


In [ ]:
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

In [ ]:
!zip -r final_model.zip final_model

  adding: final_model/ (stored 0%)
  adding: final_model/config.json (deflated 54%)
  adding: final_model/training_args.bin (deflated 54%)
  adding: final_model/tokenizer_config.json (deflated 43%)
  adding: final_model/tokenizer.json (deflated 71%)
  adding: final_model/model.safetensors (deflated 7%)


In [ ]:
from google.colab import files
files.download("final_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
0.304844,0.543991,3


{'eval_loss': 0.5439913272857666}